# MacroEcon Data Tracker — Etapa 1: Exploração de APIs
**Objetivo:** Inspecionar o schema (formato) do JSON retornado por cada API antes de construir o pipeline.

APIs exploradas:
1. GDELT (Tone Score — proxy de tensão geopolítica)
2. Yahoo Finance via `yfinance` (preço do petróleo Brent)

In [3]:
import requests
import yfinance as yf
import pandas as pd
from datetime import date, timedelta

# Datas de referência para os testes
hoje = date.today()
ontem = hoje - timedelta(days=1)
trinta_dias_atras = hoje - timedelta(days=30)

print(f'Período de teste: {trinta_dias_atras} → {ontem}')

Período de teste: 2026-03-16 → 2026-04-14


---
## 1. GDELT — Tone Score (Tensão Geopolítica)

A GDELT DOC 2.0 API retorna artigos com metadados, incluindo o **Tone score** (sentimento médio dos artigos).
- Tone negativo → cobertura de crise/conflito
- Tone positivo → cobertura de resolução/diplomacia

**Parâmetros importantes:**
- `query`: termo de busca
- `mode=artlist`: retorna lista de artigos com metadados
- `maxrecords`: máximo de 250 por chamada (sem autenticação)
- `timespan`: janela de tempo (`1d` = último dia)

In [ ]:
import time

GDELT_URL = 'https://api.gdeltproject.org/api/v2/doc/doc'

time.sleep(6)

headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

params_tone = {
    'query': 'Middle East conflict geopolitical tension',
    'mode': 'timelinetone',
    'timespan': '7d',
    'format': 'json'
}

response_tone = requests.get(GDELT_URL, params=params_tone, headers=headers)
print(f'Status HTTP: {response_tone.status_code}')

if response_tone.status_code == 200:
    tone_raw = response_tone.json()
    print(f'Chaves do JSON: {list(tone_raw.keys())}')
    print(f'\nPrimeiro registro: {tone_raw["timeline"][0]["data"][0]}')
    print(f'Total de registros horários: {len(tone_raw["timeline"][0]["data"])}')
else:
    print(f'Erro: {response_tone.status_code}')
    print(f'Resposta: {response_tone.text[:200]}')

In [22]:
# Inspecionar o schema de um artigo individual
if gdelt_raw.get('articles'):
    primeiro_artigo = gdelt_raw['articles'][0]
    print('--- Schema de um artigo ---')
    for chave, valor in primeiro_artigo.items():
        print(f'  {chave}: {repr(valor)}')
else:
    print('Nenhum artigo retornado. Verifique o timespan ou o termo de busca.')

--- Schema de um artigo ---
  url: 'https://money.kompas.com/read/2026/04/13/094500226/dari-hormuz-ke-sawah--geopolitik-pupuk-dan-ketahanan-pangan-dunia'
  url_mobile: ''
  title: 'Dari Hormuz ke Sawah : Geopolitik Pupuk dan Ketahanan Pangan Dunia'
  seendate: '20260413T031500Z'
  socialimage: 'https://asset.kompas.com/crops/nUtdQdO4xrdtdAQ5qHb2AjltSlY=/581x0:7385x4536/1200x675/filters:watermark(data/photo/2026/01/30/697c815e7ef28.png,0,-0,1)/data/photo/2024/03/05/65e66a9a46b95.jpg'
  domain: 'money.kompas.com'
  language: 'Indonesian'
  sourcecountry: 'Indonesia'


In [23]:
# Extrair o Tone Score de todos os artigos retornados
# O campo 'tone' é uma string com múltiplos valores separados por vírgula.
# Posição 0 = Tone geral (negativo = mais tenso)

artigos = gdelt_raw.get('articles', [])
registros = []

for artigo in artigos:
    tone_raw = artigo.get('tone', '')
    tone_geral = float(tone_raw.split(',')[0]) if tone_raw else None
    registros.append({
        'url': artigo.get('url'),
        'data': artigo.get('seendate'),
        'dominio': artigo.get('domain'),
        'tone_geral': tone_geral
    })

df_gdelt = pd.DataFrame(registros)
print(f'Artigos coletados: {len(df_gdelt)}')
print(f'\nTone Score médio do período: {df_gdelt["tone_geral"].mean():.2f}')
print('\nPreview:')
df_gdelt.head()

Artigos coletados: 10

Tone Score médio do período: nan

Preview:


,url,data,dominio,tone_geral
0,https://money.kompas.com/read/2026/04/13/09450...,20260413T031500Z,money.kompas.com,None
1,https://www.jawapos.com/jabodetabek/2604130124...,20260413T060000Z,jawapos.com,None
2,https://thanhnien.vn/gia-vang-hom-nay-1342026-...,20260413T021500Z,thanhnien.vn,None
3,https://www.liputan6.com/bisnis/read/6315220/r...,20260413T053000Z,liputan6.com,None
4,https://economy.okezone.com/read/2026/04/13/32...,20260413T113000Z,economy.okezone.com,None


---
## 2. Yahoo Finance via yfinance — Preço do Petróleo Brent

**Ticker do Brent:** `BZ=F` (contrato futuro Brent Crude)

Para o pipeline, usaremos apenas o **Close** (preço de fechamento do dia).

**Atenção:** fins de semana não retornam dados — bolsa fechada.

In [24]:
# --- Yahoo Finance ---
TICKER_BRENT = 'BZ=F'

brent = yf.Ticker(TICKER_BRENT)

df_brent = brent.history(start=str(trinta_dias_atras), end=str(hoje))

print(f'Tipo retornado: {type(df_brent)}')
print(f'Shape: {df_brent.shape}')
print(f'\nColunas: {list(df_brent.columns)}')
print(f'\nTipo do índice: {type(df_brent.index)}')
print('\nPreview:')
df_brent.head()

Tipo retornado: <class 'pandas.DataFrame'>
Shape: (19, 7)

Colunas: ['Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits']

Tipo do índice: <class 'pandas.DatetimeIndex'>

Preview:


,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-03-16 00:00:00-04:00,105.510002,106.510002,99.559998,100.209999,77839,0.0,0.0
2026-03-17 00:00:00-04:00,101.370003,104.980003,100.760002,103.419998,65199,0.0,0.0
2026-03-18 00:00:00-04:00,103.449997,111.860001,100.339996,107.379997,73104,0.0,0.0
2026-03-19 00:00:00-04:00,109.660004,119.120003,103.750000,108.650002,87485,0.0,0.0
2026-03-20 00:00:00-04:00,107.669998,113.110001,105.099998,112.190002,57217,0.0,0.0


In [17]:
# Isolar apenas o Close e normalizar o índice
df_brent_clean = df_brent[['Close']].copy()
df_brent_clean.index = df_brent_clean.index.date
df_brent_clean.index.name = 'data'
df_brent_clean.columns = ['preco_brent_usd']

print('Preço de fechamento — últimos 30 dias úteis:')
print(df_brent_clean)

Preço de fechamento — últimos 30 dias úteis:
            preco_brent_usd
data                       
2026-03-16       100.209999
2026-03-17       103.419998
2026-03-18       107.379997
2026-03-19       108.650002
2026-03-20       112.190002
2026-03-23        99.940002
2026-03-24       104.489998
2026-03-25       102.220001
2026-03-26       108.010002
2026-03-27       112.570000
2026-03-30       112.779999
2026-03-31       118.349998
2026-04-01       101.160004
2026-04-02       109.029999
2026-04-06       109.769997
2026-04-07       109.269997
2026-04-08        94.750000
2026-04-09        95.919998
2026-04-10        95.199997


---
## 3. Resumo: Schemas para o Pipeline

Após rodar as células acima, temos o schema real de cada fonte.
Essas informações guiam a estrutura de pastas no ADLS Gen2 (Bronze) e a lógica de transformação.

In [18]:
# Resumo dos campos que vamos persistir no ADLS Gen2 (Camada Bronze)
schema_bronze = {
    'gdelt': {
        'campos': ['seendate', 'tone_geral', 'dominio', 'url'],
        'granularidade': 'artigo por artigo',
        'frequencia': 'diária',
        'formato': 'JSON',
        'caminho_adls': 'raw/gdelt/ano/mes/dia/'
    },
    'yfinance': {
        'campos': ['Date', 'Close'],
        'granularidade': 'um valor por dia útil',
        'frequencia': 'diária (dias úteis)',
        'formato': 'JSON',
        'caminho_adls': 'raw/yfinance/ano/mes/dia/'
    }
}

for fonte, info in schema_bronze.items():
    print(f'\n[{fonte.upper()}]')
    for k, v in info.items():
        print(f'  {k}: {v}')

print('\n⚠️  PONTO DE ATENÇÃO:')
print('GDELT retorna dados de fins de semana, yfinance não.')
print('A transformação precisará de forward fill para os dias sem preço do Brent.')


[GDELT]
  campos: ['seendate', 'tone_geral', 'dominio', 'url']
  granularidade: artigo por artigo
  frequencia: diária
  formato: JSON
  caminho_adls: raw/gdelt/ano/mes/dia/

[YFINANCE]
  campos: ['Date', 'Close']
  granularidade: um valor por dia útil
  frequencia: diária (dias úteis)
  formato: JSON
  caminho_adls: raw/yfinance/ano/mes/dia/

⚠️  PONTO DE ATENÇÃO:
GDELT retorna dados de fins de semana, yfinance não.
A transformação precisará de forward fill para os dias sem preço do Brent.


In [25]:
# Inspecionar o campo tone bruto de cada artigo
for i, artigo in enumerate(gdelt_raw.get('articles', [])):
    print(f"Artigo {i}: tone bruto = {repr(artigo.get('tone'))}")
    print(f"         todas as chaves = {list(artigo.keys())}")
    print()

Artigo 0: tone bruto = None
         todas as chaves = ['url', 'url_mobile', 'title', 'seendate', 'socialimage', 'domain', 'language', 'sourcecountry']

Artigo 1: tone bruto = None
         todas as chaves = ['url', 'url_mobile', 'title', 'seendate', 'socialimage', 'domain', 'language', 'sourcecountry']

Artigo 2: tone bruto = None
         todas as chaves = ['url', 'url_mobile', 'title', 'seendate', 'socialimage', 'domain', 'language', 'sourcecountry']

Artigo 3: tone bruto = None
         todas as chaves = ['url', 'url_mobile', 'title', 'seendate', 'socialimage', 'domain', 'language', 'sourcecountry']

Artigo 4: tone bruto = None
         todas as chaves = ['url', 'url_mobile', 'title', 'seendate', 'socialimage', 'domain', 'language', 'sourcecountry']

Artigo 5: tone bruto = None
         todas as chaves = ['url', 'url_mobile', 'title', 'seendate', 'socialimage', 'domain', 'language', 'sourcecountry']

Artigo 6: tone bruto = None
         todas as chaves = ['url', 'url_mobile', 'titl

In [26]:
# Testando modo correto para obter Tone Score agregado
params_tone = {
    'query': 'Middle East conflict geopolitical tension',
    'mode': 'timelinetone',
    'timespan': '30d',
    'format': 'json'
}

response_tone = requests.get(GDELT_URL, params=params_tone)
print(f'Status HTTP: {response_tone.status_code}')

if response_tone.status_code == 200:
    tone_raw = response_tone.json()
    print(f'\nChaves do JSON: {list(tone_raw.keys())}')
    print(f'\nPrimeiro registro:')
    print(tone_raw['timeline'][0] if tone_raw.get('timeline') else 'Sem dados')
else:
    print(f'Erro: {response_tone.status_code}')

Status HTTP: 429
Erro: 429
